# ⚖️ Veredito Fase 5 — o núcleo nativo reproduz o motor Python bit a bit, em escala?

Notebook de veredito da **Fase 5 do Degrau 1** (núcleo nativo C++/nanobind em `src/mundo/native/`, backend opt-in `mundo.native_engine.NativeEngine` com a mesma assinatura do `Engine`). Protocolo **pré-registrado**:

- **Paridade (o portão):** corridas gêmeas Python × nativo com hash SHA-256 do `state_dict()` (o protocolo dos goldens da Fase 1) e do `result()` idênticos, em: clássico n=8 (120 ticks) e n=100 (48 ticks); mundo vivo completo (market_v2 + country_v2 + equip_v2) n=8 (120 ticks), n=100 (48 ticks) e n=1.000 (24 ticks); seeds 42, 7, 123; eventos de Deus (choque/descoberta/sanção); force_events políticos; restore cruzado (30 ticks num backend + `from_state` no outro + 30 ticks == corrida contínua de 60).
- **Regra de aprovação:** paridade bit-idêntica em TODAS as corridas (sem tolerância — MT19937 do CPython, Neumaier onde o motor usa `sum()`, `-ffp-contract=off`; divergência de 1 ulp já reprova e vira defeito a corrigir, não "aproximação aceitável"). Fora isso: REPROVADO/AJUSTAR.
- **Benchmark (informativo, não é portão):** `step()` cronometrado, mundo vivo ligado, n=100 e n=1.000, mediana de repetições — Python × nativo nesta máquina (Kaggle, 4 vCPU). A régua de referência é o estudo do Portão 0 e a medição local registrada no LOG (12,1× a n=100; 18,7× a n=1.000, step inteiro do mundo vivo).
- O núcleo compila NO KERNEL a partir das fontes do bundle (`mundo.native.build`, g++ do próprio Kaggle) — nenhum binário viaja no dataset.

Regra do fundador: simulação de verdade roda AQUI, no Kaggle — nunca na VM de desenvolvimento.


In [ ]:
# 1) Fonte do pacote: dataset privado do bundle mais recente
#    (mundo-autonomo-srcN). Pacote Python puro: NAO precisa de pip
#    install. LOADER RECURSIVO: o Kaggle expoe os datasets em
#    /kaggle/input/datasets/<dono>/<slug>/ com a arvore ja extraida;
#    a busca aceita qualquer 'src/mundo/__init__.py' sob /kaggle/input,
#    com fallback para o zip mundo-src.zip.
import glob, os, sys, zipfile

inits = sorted(glob.glob('/kaggle/input/**/src/mundo/__init__.py',
                         recursive=True))
zip_cands = sorted(glob.glob('/kaggle/input/**/mundo*.zip', recursive=True))
if inits:
    src_root = os.path.dirname(os.path.dirname(os.path.dirname(inits[-1])))
elif zip_cands:
    zipfile.ZipFile(zip_cands[-1]).extractall('/kaggle/working/pkg')
    src_root = '/kaggle/working/pkg'
else:
    raise SystemExit('Fonte do pacote nao encontrada (loader recursivo): '
                     'anexe o dataset pascalcoimbra/mundo-autonomo-srcN '
                     'a este kernel.')
print('pacote em', src_root)
sys.path.insert(0, os.path.join(src_root, 'src'))
import mundo
print('mundo', mundo.__version__, 'importado de', src_root)


In [ ]:
# 2) Compila o nucleo nativo a partir das fontes do bundle e confere
#    a toolchain (g++ + nanobind; nanobind vem por pip, pacote puro de
#    headers + um .cpp — a compilacao leva ~1 min).
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'nanobind'],
               check=True)
from mundo import native
from mundo.native import build
path = build.build(force=True)
print('nucleo compilado:', path)
print('native.available():', native.available())
assert native.available(), 'nucleo nativo nao carregou no kernel'


## Paridade em escala

Cada corrida roda os DOIS backends com os mesmos parâmetros e compara o hash do `state_dict()` e do `result()`. Qualquer divergência reprova.

In [ ]:
# 3) Paridade: hash de estado e de resultado, Python x nativo.
import hashlib, json

from mundo.engine import Engine
from mundo.native_engine import NativeEngine

def H(x):
    return hashlib.sha256(json.dumps(
        x, sort_keys=True, ensure_ascii=False,
        default=str).encode()).hexdigest()

LIVING = dict(market_v2=True, country_v2=True, equip_v2=True)
GOD = [
    {"tick": 5, "type": "price_shock", "commodity": "copper",
     "factor": 1.8},
    {"tick": 10, "type": "discovery", "name": "Serra Nova",
     "resource": "lithium_ore", "lat": -20.0, "lon": -45.0,
     "size_t": 9000, "grade": 0.8},
    {"tick": 15, "type": "sanction", "resource": "copper_ore",
     "duration": 8},
]
FORCE = {"force_events": [
    {"tick": 3, "country": "Chile", "event": "election"},
    {"tick": 6, "country": "Brasil", "event": "export_restriction",
     "resource": "iron_ore"},
    {"tick": 9, "country": "Argentina", "event": "disaster"},
    {"tick": 12, "country": "Peru", "event": "royalty_reform"},
    {"tick": 15, "country": "Chile", "event": "state_jv"},
    {"tick": 18, "country": "Brasil", "event": "subsidy"},
    {"tick": 21, "country": "Peru", "event": "judicial_halt"},
]}

CASES = [
    ("classico n=8 t=120", 120, dict(seed=42, n_companies=8)),
    ("classico n=100 t=48", 48, dict(seed=42, n_companies=100)),
    ("vivo n=8 t=120", 120, dict(seed=42, n_companies=8, **LIVING)),
    ("vivo n=100 t=48", 48, dict(seed=42, n_companies=100, **LIVING)),
    ("vivo n=1000 t=24", 24, dict(seed=42, n_companies=1000, **LIVING)),
    ("vivo seed 7 t=60", 60, dict(seed=7, n_companies=8, **LIVING)),
    ("vivo seed 123 t=60", 60, dict(seed=123, n_companies=8, **LIVING)),
    ("vivo+deus t=60", 60, dict(seed=7, n_companies=8,
                                god_events=GOD, **LIVING)),
    ("vivo+force t=48", 48, dict(seed=42, n_companies=8,
                                 country_params=FORCE, **LIVING)),
]

results = []
for label, ticks, kw in CASES:
    py = Engine(**kw); py.run(ticks)
    nat = NativeEngine(**kw); nat.run(ticks)
    ok = H(py.state_dict()) == H(nat.state_dict()) \
        and H(py.result()) == H(nat.result())
    results.append((label, ok))
    print(f"{label:<22} paridade: {'OK' if ok else 'DIVERGIU'}")

# restore cruzado: 30 ticks num backend, from_state no outro, +30
def steps(e, n):
    for _ in range(n):
        e.step()

restore_ok = True
for flags, label in [(dict(), 'classico'), (LIVING, 'vivo')]:
    ref = Engine(seed=42, n_companies=8, **flags); steps(ref, 60)
    a = Engine(seed=42, n_companies=8, **flags); steps(a, 30)
    b = NativeEngine.from_state(a.state_dict()); steps(b, 30)
    c = NativeEngine(seed=42, n_companies=8, **flags); steps(c, 30)
    d = Engine.from_state(c.state_dict()); steps(d, 30)
    ok = H(b.state_dict()) == H(ref.state_dict()) \
        and H(d.state_dict()) == H(ref.state_dict())
    restore_ok &= ok
    results.append((f"restore cruzado {label}", ok))
    print(f"restore cruzado {label:<10} paridade: {'OK' if ok else 'DIVERGIU'}")


## Benchmark

`step()` cronometrado com o mundo vivo ligado (mediana de repetições), no padrão do estudo do Portão 0 — informativo; o portão é a paridade.

In [ ]:
# 4) Benchmark: ticks/s Python x nativo, mundo vivo ligado.
import statistics, time

def bench(cls, n, ticks, reps):
    rates = []
    for _ in range(reps):
        e = cls(seed=42, n_companies=n, **LIVING)
        t0 = time.perf_counter()
        steps(e, ticks)
        rates.append(ticks / (time.perf_counter() - t0))
    return statistics.median(rates)

bench_rows = []
for n, ticks, reps in [(100, 60, 3), (1000, 24, 2)]:
    rp = bench(Engine, n, ticks, reps)
    rn = bench(NativeEngine, n, ticks, reps)
    bench_rows.append((n, rp, rn, rn / rp))
    print(f"n={n:<5} python {rp:9.1f} ticks/s | nativo {rn:9.1f} "
          f"ticks/s | speedup {rn / rp:.1f}x")


## O veredito

A regra é a pré-registrada no topo: **todas** as corridas de paridade bit-idênticas → Fase 5 APROVADA; qualquer divergência → REPROVADO/AJUSTAR com a corrida nomeada.

In [ ]:
# 5) VEREDITO (regra pre-registrada)
import json

failed = [label for label, ok in results if not ok]
verdict = "APROVADO" if not failed else "REPROVADO/AJUSTAR"
print("=" * 64)
print(f"VEREDITO FASE 5 (nucleo nativo): {verdict}")
if failed:
    print("corridas divergentes:", failed)
print(f"paridade: {sum(1 for _, ok in results if ok)}/{len(results)} "
      "corridas bit-identicas")
for n, rp, rn, sp in bench_rows:
    print(f"benchmark n={n}: python {rp:.1f} x nativo {rn:.1f} "
          f"ticks/s (speedup {sp:.1f}x)")
print("=" * 64)
with open('/kaggle/working/veredito-fase5.json', 'w',
          encoding='utf-8') as f:
    json.dump({"verdict": verdict,
               "parity": {label: ok for label, ok in results},
               "benchmark": [{"n": n, "python_tps": rp,
                              "native_tps": rn, "speedup": sp}
                             for n, rp, rn, sp in bench_rows]},
              f, ensure_ascii=False, indent=1)
